In [ ]:
df_final = pd.read_csv('/Users/mac/Desktop/homework/homework/MWandOccupations/data/processed/russia_rlms_datapro.csv')
df_final.columns.tolist()


In [ ]:
# ============================================================================
# RUSSIA MINIMUM WAGE AND LABOR MOBILITY STUDY - ANALYSIS PIPELINE
# ============================================================================
# Complete analysis code with event studies and outcome testing
# ============================================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import logging
from pathlib import Path
from tabulate import tabulate

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    """Global configuration settings."""
    
    # Paths
    FIGURES_DIR: Path = Path('MWandOccupations/results/figures')
    OUTPUT_DIR: Path = Path('MWandOccupations/results/logs')
    TABLES_DIR: Path = Path('MWandOccupations/results/tables')
    
    # Analysis parameters
    MIN_YEAR: int = 2000
    MAX_YEAR: int = 2024
    MW_GAP_UNIT: int = 1000
    CLUSTER_LEVEL: str = 'region'
    EVENT_WINDOW: int = 5
    PRE_PERIODS: int = 5
    POST_PERIODS: int = 10 
    
    # Treatment options
    TREATMENT_VAR: str = 'kaitz_median'
    TREATMENT_TIME_INVARIANT: bool = False
    
    # Significance levels
    ALPHA: float = 0.05
    ALPHA_STARS: Dict[float, str] = None
    
    # Plot settings
    PLOT_STYLE: str = 'seaborn-v0_8-whitegrid'
    FIGURE_SIZE: Tuple[int, int] = (12, 7)
    DPI: int = 300
    COLOR_PRE: str = 'blue'
    COLOR_POST: str = 'red'
    COLOR_REF: str = 'gray'
    
    def __post_init__(self):
        self.ALPHA_STARS = {0.01: '***', 0.05: '**', 0.10: '*'}
        self.setup()
        self._validate_treatment_var()
    
    def setup(self):
        """Create necessary directories."""
        self.FIGURES_DIR.mkdir(exist_ok=True)
        self.OUTPUT_DIR.mkdir(exist_ok=True)
        self.TABLES_DIR.mkdir(exist_ok=True)
    
    def _validate_treatment_var(self):
        """Validate treatment variable choice."""
        valid_vars = [
            'kaitz_median', 'kaitz_mean', 'kaitz_p25', 'kaitz_p10',
            'mw_gap_1000', 'bite_share_2007'
        ]
        if self.TREATMENT_VAR not in valid_vars:
            raise ValueError(f"TREATMENT_VAR must be one of {valid_vars}")

config = Config()
config.setup()

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*80)
print("RUSSIA MINIMUM WAGE AND LABOR MOBILITY STUDY")
print("METHODOLOGY PIPELINE")
print("="*80)

df_final = pd.read_csv('/Users/mac/Desktop/homework/homework/MWandOccupations/data/processed/russia_rlms_datapro.csv')

print(f"\n✅ Data loaded: {df_final.shape[0]:,} observations, {df_final.shape[1]} variables")

# Keep only relevant years
df_final = df_final[df_final['year'] >= config.MIN_YEAR].copy()
print(f"   Years: {df_final['year'].min()} to {df_final['year'].max()}")

# Check treatment variable
if config.TREATMENT_VAR not in df_final.columns:
    print(f"\n❌ Treatment variable '{config.TREATMENT_VAR}' not found!")
    similar = [c for c in df_final.columns if 'kaitz' in c or 'bite' in c or 'mw_gap' in c]
    print(f"   Available: {similar[:10]}")
    raise ValueError(f"Treatment variable {config.TREATMENT_VAR} not in data")

# Count regions for clustering note
n_regions = df_final['region'].nunique()
print(f"\n📊 Number of regions in final sample: {n_regions}")
if n_regions < 30:
    print(f"   ⚠️ Note: {n_regions} regions is below the recommended 30-40 for cluster-robust inference.")
    print(f"   Will implement wild cluster bootstrap for key outcomes as robustness check.")

print(f"\n🔧 CONFIGURATION:")
print(f"   Treatment variable: {config.TREATMENT_VAR}")
print(f"   Time-invariant: {config.TREATMENT_TIME_INVARIANT}")
print("="*80)

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def setup_logging() -> logging.Logger:
    """Configure logging."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(config.OUTPUT_DIR / 'analysis.log'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()


def format_pvalue(p: float, include_stars: bool = True) -> str:
    """Format p-value with significance stars."""
    if pd.isna(p):
        return "N/A"
    
    if include_stars:
        if p < 0.01:
            return f"{p:.4f}***"
        elif p < 0.05:
            return f"{p:.4f}**"
        elif p < 0.10:
            return f"{p:.4f}*"
    return f"{p:.4f}"


def save_figure(plt, filename: str) -> None:
    """Save figure in multiple formats."""
    path = config.FIGURES_DIR / filename
    plt.savefig(path, dpi=config.DPI, bbox_inches='tight')
    plt.savefig(path.with_suffix('.pdf'), bbox_inches='tight')
    logger.info(f"Saved figure: {filename}")


def save_table(df: pd.DataFrame, filename: str) -> None:
    """Save table to CSV and text format."""
    csv_path = config.TABLES_DIR / f"{filename}.csv"
    df.to_csv(csv_path, index=False)
    
    txt_path = config.TABLES_DIR / f"{filename}.txt"
    with open(txt_path, 'w') as f:
        f.write(tabulate(df, headers='keys', tablefmt='grid', showindex=False))
    
    logger.info(f"Saved table: {filename}")


# ============================================================================
# WILD CLUSTER BOOTSTRAP FUNCTION
# ============================================================================

def wild_cluster_bootstrap(df, outcome, treat_dummies, cluster_var='region', n_boot=999, seed=12345):
    """
    Perform wild cluster bootstrap for cluster-robust inference.
    Returns bootstrap p-value for the average post-treatment effect.
    """
    np.random.seed(seed)
    
    # Get clusters
    clusters = df[cluster_var].unique()
    n_clusters = len(clusters)
    
    # Prepare data
    formula = f"{outcome} ~ " + " + ".join(treat_dummies) + " + age + age_sq + female + C(region) + C(year)"
    original_model = smf.ols(formula, data=df).fit()
    
    # Get post-treatment dummies
    post_dummies = [d for d in treat_dummies if 'lag_' in d or d == 'treatment_treat']
    if len(post_dummies) > 0:
        original_coef = original_model.params[post_dummies].mean()
    else:
        return np.nan, np.nan
    
    # Get residuals under null
    df_null = df.copy()
    for td in treat_dummies:
        df_null[td] = 0
    
    formula_null = f"{outcome} ~ " + " + ".join(treat_dummies) + " + age + age_sq + female + C(region) + C(year)"
    null_model = smf.ols(formula_null, data=df_null).fit()
    null_residuals = null_model.resid
    
    # Bootstrap
    boot_coefs = []
    for b in range(n_boot):
        # Draw Rademacher weights
        weights = np.random.choice([-1, 1], size=n_clusters)
        weight_dict = dict(zip(clusters, weights))
        
        # Create bootstrapped outcome
        df_boot = df.copy()
        df_boot['weight'] = df_boot[cluster_var].map(weight_dict)
        df_boot[f'{outcome}_boot'] = null_model.fittedvalues + null_residuals * df_boot['weight']
        
        # Re-run regression
        formula_boot = f"{outcome}_boot ~ " + " + ".join(treat_dummies) + " + age + age_sq + female + C(region) + C(year)"
        
        try:
            boot_model = smf.ols(formula_boot, data=df_boot).fit()
            boot_coef = boot_model.params[post_dummies].mean()
            boot_coefs.append(boot_coef)
        except:
            boot_coefs.append(np.nan)
    
    # Remove NaNs
    boot_coefs = np.array([c for c in boot_coefs if not np.isnan(c)])
    
    # Calculate bootstrap p-value (two-tailed)
    if len(boot_coefs) > 0:
        boot_p = np.mean(np.abs(boot_coefs) >= np.abs(original_coef))
    else:
        boot_p = 1.0
    
    return boot_p, original_coef


# ============================================================================
# OUTCOME TESTER
# ============================================================================

class OutcomeTester:
    """
    Tests outcomes using event study methodology with configurable treatment.
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.results = []
        self.outcome_categories = {}
        self.n_regions = df['region'].nunique()
        
        logger.info(f"Using treatment variable: {config.TREATMENT_VAR}")
        logger.info(f"Number of clusters (regions): {self.n_regions}")
        if self.n_regions < 30:
            logger.warning(f"Few clusters ({self.n_regions}) detected. Will use small-cluster corrections.")
        if config.TREATMENT_TIME_INVARIANT:
            logger.info("  Treatment is time-invariant (interacted with event dummies)")
        else:
            logger.info("  Treatment is time-varying (included directly)")
        
# ============================================================================
# MODIFY THE define_outcomes METHOD IN THE OutcomeTester CLASS
# ============================================================================ 

    def define_outcomes(self) -> Dict[str, List[str]]:
        """Dynamically detect outcomes from data."""
        
        category_patterns = {
            'Professional': ['professional', 'enter_professional', 'exit_professional', 
                            'stay_professional', 'public_professional', 'private_professional',
                            'professional_narrow', 'professional_core_manager', 
                            'professional_inclusive', 'professional_strict', 'professional_edu_higher'],
            
            'Managerial': ['manager_senior', 'manager_mid', 'manager_junior', 'manager_any',
                          'manager_public', 'manager_private', 'manager_female', 'manager_male',
                          'manager_enter', 'manager_exit', 'manager_upward', 'manager_downward'],
            
            'Gender': ['stayed_male', 'stayed_female', 'stayed_mixed',
                      'male_to_female', 'female_to_male', 'male_to_mixed', 
                      'female_to_mixed', 'mixed_to_male', 'mixed_to_female',
                      'prof_male_to_female', 'prof_female_to_male'],
            
            'Labor Supply': ['informal', 'has_second_job', 'in_kind_payment', 
                            'unpaid_leave', 'wage_arrears', 'prof_informal', 'prof_second_job'],
            
            'Mobility': ['upward_mobility', 'downward_mobility', 'lateral_mobility',
                        'occ_changed_4digit', 'occ_changed_1digit',
                        'prof_upward', 'prof_downward'],
            
            'Wage': ['log_real_wage_cpi', 'real_wage_cpi', 'low_wage', 
                    'near_min_wage', 'below_min_wage', 'kaitz_individual',
                    'kaitz_median', 'prof_low_wage', 'prof_near_min'],
        }
        
        self.outcome_categories = {}
        for category, patterns in category_patterns.items():
            found = [col for col in patterns if col in self.df.columns]
            if found:
                self.outcome_categories[category] = found
        
        total = sum(len(v) for v in self.outcome_categories.values())
        logger.info(f"Found {total} outcomes across {len(self.outcome_categories)} categories")
        
        return self.outcome_categories
    
    def prepare_event_study_data(self) -> pd.DataFrame:
        """Prepare data for event study with treatment as bite measure."""
        
        df = self.df.copy()
        
        # Create event_time if needed
        if 'event_time' not in df.columns:
            df['event_time'] = df['year'] - df['first_treat_year']
        
        # Handle never-treated units
        df.loc[df['first_treat_year'].isna(), 'event_time'] = -999
        
        # Clip to reasonable range for treated units
        mask_treated = df['first_treat_year'].notna()
        df.loc[mask_treated, 'event_time'] = df.loc[mask_treated, 'event_time'].clip(
            -config.PRE_PERIODS, config.POST_PERIODS
        )
        
        df['event_time'] = pd.to_numeric(df['event_time'], errors='coerce').fillna(-999).astype(int)
        
        # Create leads and lags dummies
        for t in range(-config.PRE_PERIODS, config.POST_PERIODS + 1):
            if t < 0:
                dummy_name = f'lead_{abs(t)}'
            elif t == 0:
                dummy_name = 'treatment'
            else:
                dummy_name = f'lag_{t}'
            
            df[dummy_name] = 0
            mask = (df['first_treat_year'].notna()) & (df['event_time'] == t)
            df.loc[mask, dummy_name] = 1
            
            # Create interaction with treatment variable
            df[f'{dummy_name}_treat'] = df[dummy_name] * df[config.TREATMENT_VAR]
        
        # Create treatment indicator
        df['ever_treated'] = (~df['first_treat_year'].isna()).astype(int)
        
        logger.info(f"Created {config.PRE_PERIODS + config.POST_PERIODS + 1} event dummies")
        
        return df
        
    def save_individual_event_studies(self, results_df):
        """Save individual event study plots for key outcomes."""
        print("\n📊 Saving individual event study figures...")
        
        figure_map = {
            'log_real_wage_cpi': ('Figure14_event_study_log_wage.png', 'Log Real Wage'),
            'low_wage': ('Figure15_event_study_low_wage.png', 'Low-Wage Employment'),
            'informal': ('Figure16_event_study_informal.png', 'Informal Employment')
        }
        
        for outcome, (filename, title) in figure_map.items():
            if outcome not in results_df['outcome'].values:
                print(f"  ⚠️ {outcome} not found in results")
                continue
                
            outcome_result = next((r for r in self.results if r['outcome'] == outcome), None)
            if outcome_result is None:
                continue
                
            plot_df = outcome_result['results_df']
            
            plt.figure(figsize=(12, 7))
            
            pre = plot_df[plot_df['event_time'] < 0]
            post = plot_df[plot_df['event_time'] >= 0]
            
            # Scale: coefficient * 100 * 10 (for 0.1 bite increase)
            if len(pre) > 0:
                plt.errorbar(pre['event_time'], pre['coefficient'] * 100 * 10,
                            yerr=1.96 * pre['std_error'] * 100 * 10,
                            fmt='o-', color='blue', capsize=3, label='Pre-treatment', 
                            markersize=8, linewidth=2)
            
            if len(post) > 0:
                plt.errorbar(post['event_time'], post['coefficient'] * 100 * 10,
                            yerr=1.96 * post['std_error'] * 100 * 10,
                            fmt='s-', color='red', capsize=3, label='Post-treatment', 
                            markersize=8, linewidth=2)
            
            plt.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
            plt.axvline(x=-0.5, color='gray', linestyle='--', linewidth=2, alpha=0.7)
            
            plt.xlabel('Years Relative to Minimum Wage Increase', fontsize=12)
            plt.ylabel('Effect (percentage points)', fontsize=12)
            plt.legend(loc='best', frameon=True)
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(f"{config.FIGURES_DIR}/{filename}", dpi=300, bbox_inches='tight')
            plt.show()
            print(f"  ✓ Saved: {filename}") 

            
    def test_outcome(self, outcome: str, event_df: pd.DataFrame, run_bootstrap: bool = False) -> Optional[Dict]:
        """Run event study with treatment intensity."""
        
        if outcome not in event_df.columns:
            return None
        
        # Prepare data
        outcome_df = event_df.dropna(subset=[outcome, 'region', 'year', config.TREATMENT_VAR, 'age', 'female']).copy()
        
        if len(outcome_df) < 100:
            return None
        
        # Filter to relevant observations
        outcome_df = outcome_df[
            (outcome_df['ever_treated'] == 0) | 
            (outcome_df['event_time'].between(-config.PRE_PERIODS, config.POST_PERIODS))
        ].copy()
        
        # Create age_sq
        if 'age' in outcome_df.columns:
            outcome_df['age_sq'] = outcome_df['age'] ** 2
        
        # Build dummies list with treatment interactions
        treat_dummies = []
        for t in range(-config.PRE_PERIODS, config.POST_PERIODS + 1):
            if t < 0:
                name = f'lead_{abs(t)}_treat'
            elif t == 0:
                name = 'treatment_treat'
            else:
                name = f'lag_{t}_treat'
            if name in outcome_df.columns:
                treat_dummies.append(name)
        
        # Remove reference period (t=-1)
        reference = 'lead_1_treat'
        if reference in treat_dummies:
            treat_dummies.remove(reference)
        
        # Build formula using treatment interactions
        formula = f"{outcome} ~ " + " + ".join(treat_dummies)
        
        # Add main effect of treatment (for time-varying)
        if not config.TREATMENT_TIME_INVARIANT:
            formula += f" + {config.TREATMENT_VAR}"
        
        # Add controls
        formula += " + age + age_sq + female"
        
        # Add fixed effects - REGION + YEAR
        formula += " + C(region) + C(year)"
        
        try:
            # Run regression with clustered standard errors (use_t=True for small cluster correction)
            model = smf.ols(formula, data=outcome_df).fit(
                cov_type='cluster',
                cov_kwds={'groups': outcome_df['region']},
                use_t=True  # Use t-distribution for small clusters
            )
            
            # Extract coefficients
            results = []
            for t in range(-config.PRE_PERIODS, config.POST_PERIODS + 1):
                if t == -1:
                    continue
                    
                if t < 0:
                    name = f'lead_{abs(t)}_treat'
                elif t == 0:
                    name = 'treatment_treat'
                else:
                    name = f'lag_{t}_treat'
                
                if name in model.params.index:
                    results.append({
                        'event_time': t,
                        'coefficient': model.params[name],
                        'std_error': model.bse[name],
                        'p_value': model.pvalues[name],
                        'ci_lower': model.conf_int().loc[name][0],
                        'ci_upper': model.conf_int().loc[name][1]
                    })
            
            if not results:
                return None
            
            results_df = pd.DataFrame(results).sort_values('event_time')
             
            # Pre-trends test
            pre_data = results_df[results_df['event_time'] < 0]
            if len(pre_data) >= 2:
                pre_dummies = []
                for t in range(-config.PRE_PERIODS, 0):
                    if t == -1:
                        continue
                    if t < 0:
                        name = f'lead_{abs(t)}_treat'
                        if name in model.params.index:
                            pre_dummies.append(name)
                
                if len(pre_dummies) >= 2:
                    try:
                        f_test = model.f_test(pre_dummies)
                        pre_trend_p = f_test.pvalue
                        
                        if len(pre_data) >= 3:
                            slope, slope_p, _, _, _ = stats.linregress(
                                pre_data['event_time'], pre_data['coefficient']
                            )
                        else:
                            slope, slope_p = np.nan, np.nan
                    except Exception:
                        pre_trend_p = 1.0
                        slope, slope_p = np.nan, np.nan
                else:
                    pre_trend_p = 1.0
                    slope, slope_p = np.nan, np.nan
            else:
                pre_trend_p = 1.0
                slope, slope_p = np.nan, np.nan
                    
            # Post-treatment averages
            post_data = results_df[results_df['event_time'] >= 0]
            
            if len(post_data) > 0:
                avg_post = post_data['coefficient'].mean()
                avg_post_se = np.sqrt((post_data['std_error'] ** 2).mean())
                avg_post_p = 2 * (1 - stats.t.cdf(abs(avg_post / avg_post_se), df=self.n_regions - 1))
                
                lr_row = post_data[post_data['event_time'] == post_data['event_time'].max()]
                if len(lr_row) > 0:
                    lr_effect = lr_row['coefficient'].iloc[0]
                    lr_p = lr_row['p_value'].iloc[0]
                else:
                    lr_effect, lr_p = np.nan, np.nan
            else:
                avg_post, avg_post_p, lr_effect, lr_p = np.nan, np.nan, np.nan, np.nan
            
            # Run bootstrap for key outcomes if requested
            boot_p = np.nan
            if run_bootstrap and outcome in ['low_wage', 'exit_professional', 'log_real_wage_cpi', 'informal']:
                try:
                    boot_p, _ = wild_cluster_bootstrap(outcome_df, outcome, treat_dummies, n_boot=999)
                except Exception as e:
                    logger.debug(f"Bootstrap failed for {outcome}: {str(e)[:50]}")
            
            treatment_scale = {
                'kaitz_median': 'per 0.1 increase in bite',
                'kaitz_mean': 'per 0.1 increase in bite',
                'kaitz_p25': 'per 0.1 increase in bite',
                'kaitz_p10': 'per 0.1 increase in bite',
                'mw_gap_1000': 'per 1000 ruble increase',
                'bite_share_2007': 'per 10pp increase in affected share'
            }.get(config.TREATMENT_VAR, 'per unit increase')
            
            return {
                'outcome': outcome,
                'treatment_var': config.TREATMENT_VAR,
                'treatment_scale': treatment_scale,
                'pre_trend_p': pre_trend_p,
                'pre_trend_slope': slope,
                'pre_trend_slope_p': slope_p,
                'avg_post': avg_post,
                'avg_post_p': avg_post_p,
                'boot_p': boot_p,
                'lr_effect': lr_effect,
                'lr_p': lr_p,
                'n_obs': len(outcome_df),
                'n_clusters': self.n_regions,
                'model': model,
                'results_df': results_df
            }
            
        except Exception as e:
            logger.debug(f"Error testing {outcome}: {str(e)[:100]}")
            return None
   
    def test_all_outcomes(self) -> pd.DataFrame:
        """Test all outcomes and compile results."""
        
        self.define_outcomes()
        
        logger.info("Preparing event study data...")
        event_df = self.prepare_event_study_data()
        
        all_outcomes = []
        for category, outcomes in self.outcome_categories.items():
            all_outcomes.extend(outcomes)
        
        logger.info(f"Testing {len(all_outcomes)} outcomes...")
        logger.info(f"Note: Using t-distribution with {self.n_regions - 1} degrees of freedom for small-cluster correction")
        
        for i, outcome in enumerate(all_outcomes):
            if i % 10 == 0:
                logger.info(f"  Progress: {i}/{len(all_outcomes)}")
            
            # Run bootstrap only for key outcomes
            run_bootstrap = outcome in ['low_wage', 'exit_professional', 'log_real_wage_cpi', 'informal']
            result = self.test_outcome(outcome, event_df, run_bootstrap=run_bootstrap)
            if result:
                for cat, outs in self.outcome_categories.items():
                    if outcome in outs:
                        result['category'] = cat
                        break
                
                self.results.append(result)
        
        results_df = pd.DataFrame(self.results)
        
        if len(results_df) > 0:
            p_values = results_df['avg_post_p'].fillna(1).values
            _, p_corrected, _, _ = multipletests(
                p_values, alpha=config.ALPHA, method='fdr_bh'
            )
            results_df['avg_post_p_corrected'] = p_corrected
            results_df['significant'] = p_corrected < config.ALPHA
            
            if any(x in config.TREATMENT_VAR for x in ['kaitz', 'bite']):
                scale_factor = 0.1
            else:
                scale_factor = 1
            
            results_df['avg_post_scaled'] = results_df['avg_post'] * scale_factor
            results_df['lr_effect_scaled'] = results_df['lr_effect'] * scale_factor
            results_df['effect_magnitude_pp'] = results_df['avg_post_scaled'] * 100
            results_df['lr_magnitude_pp'] = results_df['lr_effect_scaled'] * 100
            
            results_df['effect_direction'] = results_df['avg_post'].apply(
                lambda x: 'Positive' if x > 0 else 'Negative' if x < 0 else 'Zero'
            )
        
        return results_df
    
    def create_summary_tables(self, results_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Create summary tables for output."""
        
        tables = {}
        
        table1 = results_df[[
            'category', 'outcome', 'avg_post_scaled', 'avg_post_p', 
            'avg_post_p_corrected', 'significant', 'pre_trend_p',
            'boot_p', 'lr_effect_scaled', 'lr_p', 'n_obs', 'n_clusters', 'treatment_scale'
        ]].copy()
        
        table1['avg_post_pp'] = table1['avg_post_scaled'] * 100
        table1['lr_effect_pp'] = table1['lr_effect_scaled'] * 100
        table1['avg_post_formatted'] = table1.apply(
            lambda row: f"{row['avg_post_pp']:.2f}pp {format_pvalue(row['avg_post_p'], include_stars=False)}",
            axis=1
        )
        table1['pre_trend_formatted'] = table1['pre_trend_p'].apply(
            lambda p: f"{p:.4f}" if not pd.isna(p) else "N/A"
        )
        
        tables['all_outcomes'] = table1
        
        sig_results = results_df[results_df['significant']].copy()
        if len(sig_results) > 0:
            sig_results = sig_results.sort_values('avg_post_p_corrected')
            tables['significant_outcomes'] = sig_results
        
        strong_lr = results_df[
            (results_df['lr_p'].fillna(1) < 0.10) & 
            (abs(results_df['lr_effect_scaled']).fillna(0) > 0.01)
        ].copy()
        if len(strong_lr) > 0:
            strong_lr = strong_lr.sort_values('lr_p')
            tables['strong_longrun'] = strong_lr
        
        cat_summary = results_df.groupby('category').agg({
            'outcome': 'count',
            'significant': 'sum',
            'avg_post_scaled': lambda x: x.mean() * 100,
            'avg_post_p': lambda x: (x < 0.05).sum() / len(x) * 100
        }).round(2)
        cat_summary.columns = ['N Outcomes', 'N Significant', 'Avg Effect (pp)', '% Significant (p<0.05)']
        tables['category_summary'] = cat_summary
        
        return tables
    
    def plot_top_outcomes(self, results_df: pd.DataFrame, n_top: int = 6) -> None:
        """Plot event studies for top outcomes."""
        
        top_outcomes = results_df.nsmallest(n_top, 'avg_post_p_corrected')
        
        n_cols = 3
        n_rows = (n_top + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))
        axes = axes.flatten()
        
        for i, (_, row) in enumerate(top_outcomes.iterrows()):
            if i >= len(axes):
                break
            
            outcome_result = next(r for r in self.results if r['outcome'] == row['outcome'])
            plot_df = outcome_result['results_df']
            
            ax = axes[i]
            
            pre = plot_df[plot_df['event_time'] < 0]
            post = plot_df[plot_df['event_time'] >= 0]
            
            if any(x in config.TREATMENT_VAR for x in ['kaitz', 'bite']):
                scale = 0.1 * 100
            else:
                scale = 100
            
            if len(pre) > 0:
                ax.errorbar(pre['event_time'], pre['coefficient'] * scale,
                           yerr=1.96 * pre['std_error'] * scale,
                           fmt='o-', color=config.COLOR_PRE, capsize=3,
                           label='Pre-treatment', markersize=6)
            
            if len(post) > 0:
                ax.errorbar(post['event_time'], post['coefficient'] * scale,
                           yerr=1.96 * post['std_error'] * scale,
                           fmt='s-', color=config.COLOR_POST, capsize=3,
                           label='Post-treatment', markersize=6)
            
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
            ax.axvline(x=-0.5, color=config.COLOR_REF, linestyle='--', linewidth=1, alpha=0.7)
            
            pre_trend_p = row['pre_trend_p']
            if not pd.isna(pre_trend_p):
                color = 'green' if pre_trend_p > 0.05 else 'red'
                ax.text(0.02, 0.98, f'Pre-trend p={pre_trend_p:.4f}',
                       transform=ax.transAxes, color=color, fontsize=9,
                       verticalalignment='top')
            
            treatment_label = {
                'kaitz_median': ' (per 0.1↑ bite)',
                'kaitz_mean': ' (per 0.1↑ bite)',
                'kaitz_p25': ' (per 0.1↑ bite)',
                'kaitz_p10': ' (per 0.1↑ bite)',
                'mw_gap_1000': ' (per 1000₽)',
                'bite_share_2007': ' (per 10pp↑)'
            }.get(config.TREATMENT_VAR, '')
            
            ax.set_xlabel('Years Relative to Treatment', fontsize=9)
            ax.set_ylabel('Effect (pp)', fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8, loc='lower right')
        
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        
        plt.tight_layout()
        
        save_figure(plt, f'top_outcomes_{config.TREATMENT_VAR}.png')
        plt.show()


# ============================================================================
# ANALYSIS PIPELINE
# ============================================================================

class AnalysisPipeline:
    """Main pipeline with proven methodology."""
    
    def __init__(self, df_final: pd.DataFrame):
        self.df_final = df_final.copy()
        self.outcome_tester = OutcomeTester(self.df_final)
        self.results = {}
        self.tables = {}
        
    def run_all(self) -> None:
        """Run complete analysis pipeline."""
        logger.info("="*80)
        logger.info("STARTING METHODOLOGY ANALYSIS")
        logger.info(f"Treatment variable: {config.TREATMENT_VAR}")
        logger.info("="*80)
        
        # Step 1: Test all outcomes
        logger.info("\n📊 Step 1: Testing all outcomes...")
        results_df = self.outcome_tester.test_all_outcomes()
        
        if len(results_df) == 0:
            logger.error("No results generated!")
            return
        
        self.results['full'] = results_df
        
        # Step 2: Create summary tables
        logger.info("\n📊 Step 2: Creating summary tables...")
        self.tables = self.outcome_tester.create_summary_tables(results_df)
        
        # Step 3: Save all tables
        logger.info("\n📊 Step 3: Saving tables...")
        for name, table in self.tables.items():
            save_table(table, f"table_{name}_{config.TREATMENT_VAR}")
        
        # Step 4: Generate individual-level plots
        logger.info("\n📊 Step 4: Generating individual-level plots...")
        self.outcome_tester.plot_top_outcomes(results_df) 
        self.outcome_tester.save_individual_event_studies(results_df)
        
        # Step 5: Print summary
        self._print_summary()
        
        logger.info("\n" + "="*80)
        logger.info("✅ ANALYSIS COMPLETE")
        logger.info("="*80)
    
    def _print_summary(self) -> None:
        """Print analysis summary."""
        
        results_df = self.results['full']
        
        print("\n" + "="*100)
        print(f"ANALYSIS SUMMARY - Treatment: {config.TREATMENT_VAR}")
        print("="*100)
        
        print(f"\n📊 Total outcomes tested: {len(results_df)}")
        print(f"   Significant after FDR correction: {results_df['significant'].sum()}")
        
        # Print cluster information
        if 'n_clusters' in results_df.columns:
            n_clusters = results_df['n_clusters'].iloc[0]
            print(f"\n📊 Clustering: {n_clusters} regions")
            if n_clusters < 30:
                print(f"   ⚠️ Few clusters ({n_clusters}) - using t-distribution with {n_clusters-1} df")
        
        print("\n📈 Effect size distribution (scaled for interpretation):")
        if 'effect_magnitude_pp' in results_df.columns:
            effect_sizes = results_df['effect_magnitude_pp'].dropna()
            print(f"   Mean: {effect_sizes.mean():.2f}pp")
            print(f"   Std:  {effect_sizes.std():.2f}pp")
            print(f"   Min:  {effect_sizes.min():.2f}pp")
            print(f"   Max:  {effect_sizes.max():.2f}pp")
        
        print("\n📋 Results by category:")
        if 'category_summary' in self.tables:
            cat_summary = self.tables['category_summary']
            print(tabulate(cat_summary, headers='keys', tablefmt='simple'))
        
        if 'significant_outcomes' in self.tables:
            sig_table = self.tables['significant_outcomes']
            print("\n🎯 Significant outcomes (FDR q<0.05):")
            for _, row in sig_table.iterrows():
                effect_pp = row.get('effect_magnitude_pp', row['avg_post_scaled'] * 100)
                scale_text = row.get('treatment_scale', 'per unit')
                boot_text = f" (boot p={row['boot_p']:.3f})" if pd.notna(row.get('boot_p')) else ""
                print(f"   • {row['outcome']}: {effect_pp:.2f}pp {scale_text} (p={row['avg_post_p_corrected']:.4f}){boot_text}")
        
        if 'strong_longrun' in self.tables:
            lr_table = self.tables['strong_longrun']
            print("\n💪 Strong long-run effects (p<0.10, |effect|>1pp):")
            for _, row in lr_table.iterrows():
                pre_warn = "⚠️" if row['pre_trend_p'] < 0.05 else "✅"
                lr_pp = row.get('lr_magnitude_pp', row['lr_effect_scaled'] * 100)
                scale_text = row.get('treatment_scale', 'per unit')
                print(f"   {pre_warn} {row['outcome']}: {lr_pp:.2f}pp {scale_text} (p={row['lr_p']:.4f})")
        
        print("\n" + "="*100)
        print(f"📁 Tables saved to: {config.TABLES_DIR}/")
        print(f"📁 Figures saved to: {config.FIGURES_DIR}/")
        print("="*100)
 
# ============================================================================
# ADDITIONAL FIGURES FOR THESIS NARRATIVE
# ============================================================================

class ThesisFigures:
    """Generate additional figures to support thesis narrative."""
    
    def __init__(self, df: pd.DataFrame, outcome_tester: OutcomeTester, results_df: pd.DataFrame):
        self.df = df
        self.outcome_tester = outcome_tester
        self.results_df = results_df
        self.fig_dir = config.FIGURES_DIR
        
    def generate_all(self):
        """Generate all thesis figures."""
        print("\n" + "="*80)
        print("GENERATING THESIS NARRATIVE FIGURES")
        print("="*80)
        
        self.figure_causal_summary()
        self.figure_within_vs_between()
        self.figure_pre_trends_dashboard()
        self.figure_results_heatmap()
        self.figure_adjustment_mechanisms()
        self.figure_causal_confidence()
        
        print("\n✅ All thesis narrative figures generated!")
        print(f"   Saved to: {self.fig_dir}/")
    
    def figure_causal_summary(self):
        """Figure 17: Causal Effects Summary - Your cleanest results"""
        
        # Extract key causal outcomes (those with pre-trend p > 0.05 and significant)
        causal_outcomes = self.results_df[
            (self.results_df['pre_trend_p'] > 0.05) & 
            (self.results_df['significant'] == True)
        ].copy()
        
        if len(causal_outcomes) == 0:
            print("  ⚠️ No causal outcomes found with pre-trend p>0.05")
            return
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        outcomes = causal_outcomes['outcome'].tolist()
        effects = causal_outcomes['effect_magnitude_pp'].tolist()
        errors = causal_outcomes.apply(
            lambda row: abs(row['avg_post_scaled'] * 100 * 0.3),  # Approx SE
            axis=1
        ).tolist()
        
        # Color code: green for positive, red for negative
        colors = ['#44aa44' if e > 0 else '#ff4444' for e in effects]
        
        y_pos = np.arange(len(outcomes))
        bars = ax.barh(y_pos, effects, xerr=errors, color=colors, 
                      alpha=0.8, edgecolor='black', linewidth=1.5,
                      capsize=5, error_kw={'linewidth': 2})
        
        # Add value labels
        for i, (bar, effect, error) in enumerate(zip(bars, effects, errors)):
            label_x = effect + error + 0.5 if effect > 0 else effect - error - 1.5
            ax.text(label_x, bar.get_y() + bar.get_height()/2,
                   f'{effect:.1f}pp', ha='center', va='center',
                   fontweight='bold', fontsize=11,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9))
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels([o.replace('_', ' ').title() for o in outcomes], fontsize=12)
        ax.set_xlabel('Effect Size (percentage points)', fontweight='bold', fontsize=14)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add annotation about interpretation
        treatment_scale = self.results_df['treatment_scale'].iloc[0]
        ax.text(0.98, 0.02, f'Effect shown: {treatment_scale}', 
               transform=ax.transAxes, ha='right', fontsize=10,
               bbox=dict(boxstyle="round", facecolor='lightyellow', alpha=0.8))
        
        # Add cluster information
        n_clusters = self.results_df['n_clusters'].iloc[0] if 'n_clusters' in self.results_df.columns else '?'
        ax.text(0.02, 0.02, f'Clustered by {n_clusters} regions (t-distribution)', 
               transform=ax.transAxes, ha='left', fontsize=9,
               bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure17_causal_summary.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure17_causal_summary.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure17_causal_summary.png")
    
    def figure_within_vs_between(self):
        """Figure 18: Within-Occupation vs Between-Occupation Adjustment"""
        
        # Group outcomes by adjustment channel
        within_occupation = [
            'log_real_wage_cpi', 'informal', 'low_wage', 'below_min_wage'
        ]
        between_occupation = [
            'occ_changed_4digit', 'occ_changed_1digit', 
            'upward_mobility', 'downward_mobility', 'lateral_mobility'
        ]
        
        # Calculate average effects
        within_effects = []
        for outcome in within_occupation:
            row = self.results_df[self.results_df['outcome'] == outcome]
            if len(row) > 0:
                within_effects.append(abs(row['effect_magnitude_pp'].iloc[0]))
        
        between_effects = []
        for outcome in between_occupation:
            row = self.results_df[self.results_df['outcome'] == outcome]
            if len(row) > 0:
                between_effects.append(abs(row['effect_magnitude_pp'].iloc[0]))
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Left: Average effect magnitude
        avg_within = np.mean(within_effects) if within_effects else 0
        avg_between = np.mean(between_effects) if between_effects else 0
        
        bars = ax1.bar(['Within Occupation', 'Between Occupation'], 
                      [avg_within, avg_between],
                      color=['#2ca02c', '#d62728'], alpha=0.8,
                      edgecolor='black', linewidth=1.5)
        
        for bar, val in zip(bars, [avg_within, avg_between]):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.2,
                    f'{val:.2f}pp', ha='center', va='bottom',
                    fontweight='bold', fontsize=12)
        
        ax1.set_ylabel('Average Effect Magnitude (pp)', fontweight='bold', fontsize=12)
        ax1.set_title('(a) Where Does Adjustment Happen?', fontweight='bold', fontsize=14)
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Right: Share of significant outcomes
        within_sig = sum(1 for o in within_occupation 
                        if len(self.results_df[self.results_df['outcome'] == o]) > 0 and
                        self.results_df[self.results_df['outcome'] == o]['significant'].iloc[0])
        within_total = len([o for o in within_occupation 
                          if o in self.results_df['outcome'].values])
        
        between_sig = sum(1 for o in between_occupation 
                         if len(self.results_df[self.results_df['outcome'] == o]) > 0 and
                         self.results_df[self.results_df['outcome'] == o]['significant'].iloc[0])
        between_total = len([o for o in between_occupation 
                           if o in self.results_df['outcome'].values])
        
        sig_share_within = (within_sig / within_total * 100) if within_total > 0 else 0
        sig_share_between = (between_sig / between_total * 100) if between_total > 0 else 0
        
        bars2 = ax2.bar(['Within Occupation', 'Between Occupation'],
                        [sig_share_within, sig_share_between],
                        color=['#2ca02c', '#d62728'], alpha=0.8,
                        edgecolor='black', linewidth=1.5)
        
        for bar, val in zip(bars2, [sig_share_within, sig_share_between]):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
                    f'{val:.0f}%', ha='center', va='bottom',
                    fontweight='bold', fontsize=12)
        
        ax2.set_ylabel('Significant Outcomes (%)', fontweight='bold', fontsize=12)
        ax2.set_title('(b) Share of Outcomes Significant', fontweight='bold', fontsize=14)
        ax2.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure18_within_vs_between.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure18_within_vs_between.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure18_within_vs_between.png")
    
    def figure_pre_trends_dashboard(self):
        """Figure 19: Pre-Trends Dashboard - Transparency about identification"""
        
        # Select key outcomes
        key_outcomes = [
            'log_real_wage_cpi', 'low_wage', 'informal', 
            'below_min_wage', 'exit_professional', 'occ_changed_4digit'
        ]
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, outcome in enumerate(key_outcomes):
            if idx >= len(axes):
                break
                
            ax = axes[idx]
            
            # Find outcome result
            outcome_result = next((r for r in self.outcome_tester.results 
                                 if r['outcome'] == outcome), None)
            
            if outcome_result is None:
                ax.text(0.5, 0.5, f'{outcome}\nNot found', ha='center', va='center', fontsize=12)
                continue
            
            plot_df = outcome_result['results_df']
            pre = plot_df[plot_df['event_time'] < 0]
            post = plot_df[plot_df['event_time'] >= 0]
            
            # Plot pre-trend coefficients
            if len(pre) > 0:
                ax.errorbar(pre['event_time'], pre['coefficient'] * 100 * 10,
                           yerr=1.96 * pre['std_error'] * 100 * 10,
                           fmt='o-', color='blue', capsize=3,
                           label='Pre-treatment', markersize=6)
            
            # Plot post-treatment
            if len(post) > 0:
                ax.errorbar(post['event_time'], post['coefficient'] * 100 * 10,
                           yerr=1.96 * post['std_error'] * 100 * 10,
                           fmt='s-', color='red', capsize=3,
                           label='Post-treatment', markersize=6)
            
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
            ax.axvline(x=-0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
            
            # Get pre-trend p-value
            row = self.results_df[self.results_df['outcome'] == outcome]
            pre_trend_p = row['pre_trend_p'].iloc[0] if len(row) > 0 else 1.0
            
            # Color code based on pre-trend test
            if pre_trend_p > 0.05:
                color = 'green'
                status = '✅ Parallel trends hold'
            else:
                color = 'red'
                status = '⚠️ Pre-trends present'
            
            ax.text(0.02, 0.98, f'p={pre_trend_p:.4f}\n{status}', 
                   transform=ax.transAxes, color=color, fontsize=9,
                   verticalalignment='top',
                   bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
            
            ax.set_xlabel('Years', fontsize=8)
            ax.set_ylabel('Effect (pp)', fontsize=8)
            ax.grid(True, alpha=0.3)
            
            if idx == 0:
                ax.legend(fontsize=8, loc='lower right')
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure19_pre_trends_dashboard.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure19_pre_trends_dashboard.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure19_pre_trends_dashboard.png")
    
    def figure_results_heatmap(self):
        """Figure 20: Results Summary Heatmap"""
        
        # Select outcomes for heatmap
        outcomes_to_show = [
            'log_real_wage_cpi', 'low_wage', 'informal', 
            'below_min_wage', 'exit_professional', 'occ_changed_4digit',
            'upward_mobility', 'downward_mobility', 'professional'
        ]
        
        # Filter to outcomes that exist
        plot_outcomes = []
        effect_sizes = []
        p_values = []
        pre_trend_p = []
        
        for outcome in outcomes_to_show:
            row = self.results_df[self.results_df['outcome'] == outcome]
            if len(row) > 0:
                plot_outcomes.append(outcome)
                effect_sizes.append(row['effect_magnitude_pp'].iloc[0])
                p_values.append(row['avg_post_p_corrected'].iloc[0])
                pre_trend_p.append(row['pre_trend_p'].iloc[0])
        
        if not plot_outcomes:
            return
        
        # Create matrices for heatmap
        data = np.array([
            effect_sizes,
            [-np.log10(p) if p > 0 else 3 for p in p_values],  # -log10 p-value
            [1 if p > 0.05 else 0 for p in pre_trend_p]  # 1 if parallel trends hold
        ])
        
        fig, ax = plt.subplots(figsize=(12, 5))
        
        # Create custom colormap
        im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=-5, vmax=5)
        
        # Set labels
        ax.set_xticks(np.arange(len(plot_outcomes)))
        ax.set_yticks([0, 1, 2])
        ax.set_xticklabels([o.replace('_', ' ').title() for o in plot_outcomes], 
                          rotation=45, ha='right', fontsize=10)
        ax.set_yticklabels(['Effect Size (pp)', 'Significance\n(-log10 p)', 'Parallel Trends\n(1=Yes)'], 
                          fontweight='bold', fontsize=10)
        
        # Add text annotations
        for i in range(3):
            for j in range(len(plot_outcomes)):
                if i == 0:
                    text = f'{data[i,j]:.1f}'
                elif i == 1:
                    text = f'{data[i,j]:.1f}'
                else:
                    text = '✓' if data[i,j] == 1 else '✗'
                
                ax.text(j, i, text, ha='center', va='center',
                       fontweight='bold', fontsize=10,
                       color='white' if abs(data[i,j]) > 2 else 'black')
        
        plt.colorbar(im, ax=ax, label='Effect Size (pp) / -log10(p)')
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure20_results_heatmap.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure20_results_heatmap.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure20_results_heatmap.png")
    
    def figure_adjustment_mechanisms(self):
        """Figure 21: Adjustment Mechanisms - How workers respond"""
        
        # Group outcomes by mechanism
        mechanisms = {
            'Wage Gains': ['log_real_wage_cpi'],
            'Formalization': ['informal'],
            'Low-Wage Exit': ['low_wage'],
            'Professional Exit': ['exit_professional'],
            'Occupational Mobility': ['occ_changed_4digit', 'occ_changed_1digit'],
            'Gender Segregation': ['stayed_male', 'stayed_female'] if 'stayed_male' in self.results_df['outcome'].values else []
        }
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        mechanism_names = []
        avg_effects = []
        colors = []
        
        for mech_name, outcomes in mechanisms.items():
            effects = []
            for outcome in outcomes:
                row = self.results_df[self.results_df['outcome'] == outcome]
                if len(row) > 0:
                    effects.append(abs(row['effect_magnitude_pp'].iloc[0]))
            
            if effects:
                mechanism_names.append(mech_name)
                avg_effects.append(np.mean(effects))
                
                # Color based on mechanism type
                if mech_name in ['Wage Gains', 'Formalization', 'Low-Wage Exit']:
                    colors.append('#2ca02c')  # Green - within-occupation
                elif mech_name == 'Professional Exit':
                    colors.append('#ff7f0e')  # Orange - mixed
                else:
                    colors.append('#d62728')  # Red - no effect/between
        
        y_pos = np.arange(len(mechanism_names))
        bars = ax.barh(y_pos, avg_effects, color=colors, alpha=0.8,
                      edgecolor='black', linewidth=1.5)
        
        # Add value labels
        for bar, effect in zip(bars, avg_effects):
            ax.text(effect + 0.1, bar.get_y() + bar.get_height()/2,
                   f'{effect:.2f}pp', ha='left', va='center',
                   fontweight='bold', fontsize=11)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(mechanism_names, fontweight='bold', fontsize=12)
        ax.set_xlabel('Average Effect Magnitude (percentage points)', fontweight='bold', fontsize=14)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add annotation
        ax.text(0.98, 0.02, 'Green = Within-Occupation Adjustment\nRed = No Effect/Between', 
               transform=ax.transAxes, ha='right', fontsize=10,
               bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure21_adjustment_mechanisms.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure21_adjustment_mechanisms.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure21_adjustment_mechanisms.png")
    
    def figure_causal_confidence(self):
        """Figure 22: Confidence in Causal Claims"""
        
        # Classify outcomes by causal confidence
        causal_confident = []  # pre-trend p>0.05 & significant
        causal_suggestive = []  # pre-trend p<0.05 but significant & large
        null_clear = []  # not significant, well-powered
        
        for _, row in self.results_df.iterrows():
            outcome = row['outcome']
            effect = abs(row['effect_magnitude_pp'])
            sig = row['significant']
            pre_p = row['pre_trend_p']
            
            if sig and pre_p > 0.05:
                causal_confident.append(outcome)
            elif sig and pre_p <= 0.05:
                causal_suggestive.append(outcome)
            elif not sig and effect < 1.0:
                null_clear.append(outcome)
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(10, 6))
        
        categories = ['Causal\n(Parallel Trends + Significant)', 
                      'Suggestive\n(Significant but Pre-Trends)',
                      'Null\n(No Effect, Well-Powered)']
        counts = [len(causal_confident), len(causal_suggestive), len(null_clear)]
        colors = ['#2ca02c', '#ff7f0e', '#1f77b4']
        
        bars = ax.bar(categories, counts, color=colors, alpha=0.8,
                     edgecolor='black', linewidth=2)
        
        # Add value labels
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                   f'{count} outcomes', ha='center', va='bottom',
                   fontweight='bold', fontsize=12)
        
        # Add example outcomes
        if causal_confident:
            examples = ', '.join([o.replace('_', ' ').title()[:15] for o in causal_confident[:2]])
            ax.text(0, counts[0]/2, f'e.g., {examples}', 
                   ha='center', va='center', fontsize=9,
                   bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        ax.set_ylabel('Number of Outcomes', fontweight='bold', fontsize=14)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure22_causal_confidence.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure22_causal_confidence.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure22_causal_confidence.png")


# ============================================================================
# MODIFIED MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function with thesis figures."""
    
    # Run pipeline
    pipeline = AnalysisPipeline(df_final)
    pipeline.run_all()
    
    # Generate thesis narrative figures
    if pipeline.results and 'full' in pipeline.results:
        thesis_figs = ThesisFigures(
            df_final, 
            pipeline.outcome_tester, 
            pipeline.results['full']
        )
        thesis_figs.generate_all()


if __name__ == "__main__":
    main()

In [15]:
# Load results
results_df = pd.read_csv('MWandOccupations/results/tables/table_all_outcomes_kaitz_median.csv')
 

In [16]:
results_df[results_df.significant == True]

,category,outcome,avg_post_scaled,avg_post_p,avg_post_p_corrected,significant,pre_trend_p,boot_p,lr_effect_scaled,lr_p,n_obs,n_clusters,treatment_scale,avg_post_pp,lr_effect_pp,avg_post_formatted,pre_trend_formatted
2,Professional,exit_professional,0.003928,0.001612,0.014182,True,0.499119,0.236236,0.003932,0.001670,168753,32,per 0.1 increase in bite,0.392793,0.393177,0.39pp 0.0016,0.4991
30,Labor Supply,informal,-0.023446,0.001610,0.014182,True,0.005524,0.727728,-0.025440,0.002531,168753,32,per 0.1 increase in bite,-2.344569,-2.543974,-2.34pp 0.0016,0.0055
37,Wage,log_real_wage_cpi,0.130328,0.000002,0.000074,True,0.000449,0.043043,0.154402,0.000001,168753,32,per 0.1 increase in bite,13.032848,15.440210,13.03pp 0.0000,0.0004
39,Wage,low_wage,-0.051132,0.000075,0.001658,True,0.416127,0.000000,-0.057209,0.000100,168753,32,per 0.1 increase in bite,-5.113194,-5.720868,-5.11pp 0.0001,0.4161
41,Wage,below_min_wage,0.023406,0.000264,0.003866,True,0.000200,NaN,0.025775,0.000418,168753,32,per 0.1 increase in bite,2.340591,2.577489,2.34pp 0.0003,0.0002


In [17]:
causal_df = results_df[
    (results_df['significant'] == True) & 
    (results_df['pre_trend_p'] > 0.05)
].copy()

# Sort by effect magnitude
causal_df = causal_df.sort_values('avg_post_pp', ascending=False)
causal_df

,category,outcome,avg_post_scaled,avg_post_p,avg_post_p_corrected,significant,pre_trend_p,boot_p,lr_effect_scaled,lr_p,n_obs,n_clusters,treatment_scale,avg_post_pp,lr_effect_pp,avg_post_formatted,pre_trend_formatted
2,Professional,exit_professional,0.003928,0.001612,0.014182,True,0.499119,0.236236,0.003932,0.00167,168753,32,per 0.1 increase in bite,0.392793,0.393177,0.39pp 0.0016,0.4991
39,Wage,low_wage,-0.051132,0.000075,0.001658,True,0.416127,0.000000,-0.057209,0.00010,168753,32,per 0.1 increase in bite,-5.113194,-5.720868,-5.11pp 0.0001,0.4161


In [ ]:
#!ls MWandOccupations/results/figures

In [ ]:
# ============================================================================
# RUSSIA MINIMUM WAGE AND LABOR MOBILITY STUDY - ANALYSIS PIPELINE
# ============================================================================
# Complete analysis code with event studies and outcome testing
# ============================================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import logging
from pathlib import Path
from tabulate import tabulate

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    """Global configuration settings."""
    
    # Paths
    FIGURES_DIR: Path = Path('MWandOccupations/results/figures')
    OUTPUT_DIR: Path = Path('MWandOccupations/results/logs')
    TABLES_DIR: Path = Path('MWandOccupations/results/tables')
    
    # Analysis parameters
    MIN_YEAR: int = 2000
    MAX_YEAR: int = 2024
    MW_GAP_UNIT: int = 1000
    CLUSTER_LEVEL: str = 'region'
    EVENT_WINDOW: int = 5
    PRE_PERIODS: int = 5
    POST_PERIODS: int = 10 
    
    # Treatment options
    TREATMENT_VAR: str = 'kaitz_median'
    TREATMENT_TIME_INVARIANT: bool = False
    
    # Significance levels
    ALPHA: float = 0.05
    ALPHA_STARS: Dict[float, str] = None
    
    # Plot settings
    PLOT_STYLE: str = 'seaborn-v0_8-whitegrid'
    FIGURE_SIZE: Tuple[int, int] = (12, 7)
    DPI: int = 300
    COLOR_PRE: str = 'blue'
    COLOR_POST: str = 'red'
    COLOR_REF: str = 'gray'
    
    def __post_init__(self):
        self.ALPHA_STARS = {0.01: '***', 0.05: '**', 0.10: '*'}
        self.setup()
        self._validate_treatment_var()
    
    def setup(self):
        """Create necessary directories."""
        self.FIGURES_DIR.mkdir(exist_ok=True)
        self.OUTPUT_DIR.mkdir(exist_ok=True)
        self.TABLES_DIR.mkdir(exist_ok=True)
    
    def _validate_treatment_var(self):
        """Validate treatment variable choice."""
        valid_vars = [
            'kaitz_median', 'kaitz_mean', 'kaitz_p25', 'kaitz_p10',
            'mw_gap_1000', 'bite_share_2007'
        ]
        if self.TREATMENT_VAR not in valid_vars:
            raise ValueError(f"TREATMENT_VAR must be one of {valid_vars}")

config = Config()
config.setup()

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*80)
print("RUSSIA MINIMUM WAGE AND LABOR MOBILITY STUDY")
print("METHODOLOGY PIPELINE")
print("="*80)

df_final = pd.read_csv('/Users/mac/Desktop/homework/homework/MWandOccupations/data/processed/russia_rlms_datapro.csv')

print(f"\n✅ Data loaded: {df_final.shape[0]:,} observations, {df_final.shape[1]} variables")

# Keep only relevant years
df_final = df_final[df_final['year'] >= config.MIN_YEAR].copy()
print(f"   Years: {df_final['year'].min()} to {df_final['year'].max()}")

# Check treatment variable
if config.TREATMENT_VAR not in df_final.columns:
    print(f"\n❌ Treatment variable '{config.TREATMENT_VAR}' not found!")
    similar = [c for c in df_final.columns if 'kaitz' in c or 'bite' in c or 'mw_gap' in c]
    print(f"   Available: {similar[:10]}")
    raise ValueError(f"Treatment variable {config.TREATMENT_VAR} not in data")

print(f"\n🔧 CONFIGURATION:")
print(f"   Treatment variable: {config.TREATMENT_VAR}")
print(f"   Time-invariant: {config.TREATMENT_TIME_INVARIANT}")
print("="*80)

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def setup_logging() -> logging.Logger:
    """Configure logging."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(config.OUTPUT_DIR / 'analysis.log'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()


def format_pvalue(p: float, include_stars: bool = True) -> str:
    """Format p-value with significance stars."""
    if pd.isna(p):
        return "N/A"
    
    if include_stars:
        if p < 0.01:
            return f"{p:.4f}***"
        elif p < 0.05:
            return f"{p:.4f}**"
        elif p < 0.10:
            return f"{p:.4f}*"
    return f"{p:.4f}"


def save_figure(plt, filename: str) -> None:
    """Save figure in multiple formats."""
    path = config.FIGURES_DIR / filename
    plt.savefig(path, dpi=config.DPI, bbox_inches='tight')
    plt.savefig(path.with_suffix('.pdf'), bbox_inches='tight')
    logger.info(f"Saved figure: {filename}")


def save_table(df: pd.DataFrame, filename: str) -> None:
    """Save table to CSV and text format."""
    csv_path = config.TABLES_DIR / f"{filename}.csv"
    df.to_csv(csv_path, index=False)
    
    txt_path = config.TABLES_DIR / f"{filename}.txt"
    with open(txt_path, 'w') as f:
        f.write(tabulate(df, headers='keys', tablefmt='grid', showindex=False))
    
    logger.info(f"Saved table: {filename}")


# ============================================================================
# OUTCOME TESTER
# ============================================================================

class OutcomeTester:
    """
    Tests outcomes using event study methodology with configurable treatment.
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.results = []
        self.outcome_categories = {}
        
        logger.info(f"Using treatment variable: {config.TREATMENT_VAR}")
        if config.TREATMENT_TIME_INVARIANT:
            logger.info("  Treatment is time-invariant (interacted with event dummies)")
        else:
            logger.info("  Treatment is time-varying (included directly)")
        
# ============================================================================
# MODIFY THE define_outcomes METHOD IN THE OutcomeTester CLASS
# ============================================================================ 

    def define_outcomes(self) -> Dict[str, List[str]]:
        """Dynamically detect outcomes from data."""
        
        category_patterns = {
            'Professional': ['professional', 'enter_professional', 'exit_professional', 
                            'stay_professional', 'public_professional', 'private_professional',
                            'professional_narrow', 'professional_core_manager', 
                            'professional_inclusive', 'professional_strict', 'professional_edu_higher'],
            
            'Managerial': ['manager_senior', 'manager_mid', 'manager_junior', 'manager_any',
                          'manager_public', 'manager_private', 'manager_female', 'manager_male',
                          'manager_enter', 'manager_exit', 'manager_upward', 'manager_downward'],
            
            'Gender': ['stayed_male', 'stayed_female', 'stayed_mixed',
                      'male_to_female', 'female_to_male', 'male_to_mixed', 
                      'female_to_mixed', 'mixed_to_male', 'mixed_to_female',
                      'prof_male_to_female', 'prof_female_to_male'],
            
            'Labor Supply': ['informal', 'has_second_job', 'in_kind_payment', 
                            'unpaid_leave', 'wage_arrears', 'prof_informal', 'prof_second_job'],
            
            'Mobility': ['upward_mobility', 'downward_mobility', 'lateral_mobility',
                        'occ_changed_4digit', 'occ_changed_1digit',
                        'prof_upward', 'prof_downward'],
            
            'Wage': ['log_real_wage_cpi', 'real_wage_cpi', 'low_wage', 
                    'near_min_wage', 'below_min_wage', 'kaitz_individual',
                    'kaitz_median', 'prof_low_wage', 'prof_near_min'],
            
            # ========== ADD THESE NEW CATEGORIES ==========
            
            #'Health': ['pain', 'pain_any', 'pain_severe',
                      #'anxiety_depression', 'anxiety_any', 'anxiety_severe', 
                      #'poor_health'],
            
            #'Educational_Mismatch': ['overeducated', 'undereducated', 'matched', 'mismatch_years'],
            
            #'Mismatch_Dynamics': ['enter_overeducated', 'exit_overeducated', 
             #                    'enter_undereducated', 'exit_undereducated',
             #                    'stay_matched', 'stay_mismatched'],
            
            #'Reporting_Style': ['reporting_style']
        }
        
        self.outcome_categories = {}
        for category, patterns in category_patterns.items():
            found = [col for col in patterns if col in self.df.columns]
            if found:
                self.outcome_categories[category] = found
        
        total = sum(len(v) for v in self.outcome_categories.values())
        logger.info(f"Found {total} outcomes across {len(self.outcome_categories)} categories")
        
        # Print which health/mismatch outcomes were found
        if 'Health' in self.outcome_categories:
            logger.info(f"  Health outcomes: {len(self.outcome_categories['Health'])} found")
        if 'Educational_Mismatch' in self.outcome_categories:
            logger.info(f"  Mismatch outcomes: {len(self.outcome_categories['Educational_Mismatch'])} found")
        if 'Mismatch_Dynamics' in self.outcome_categories:
            logger.info(f"  Mismatch dynamics: {len(self.outcome_categories['Mismatch_Dynamics'])} found")
        
        return self.outcome_categories
    
    def prepare_event_study_data(self) -> pd.DataFrame:
        """Prepare data for event study with treatment as bite measure."""
        
        df = self.df.copy()
        
        # Create event_time if needed
        if 'event_time' not in df.columns:
            df['event_time'] = df['year'] - df['first_treat_year']
        
        # Handle never-treated units
        df.loc[df['first_treat_year'].isna(), 'event_time'] = -999
        
        # Clip to reasonable range for treated units
        mask_treated = df['first_treat_year'].notna()
        df.loc[mask_treated, 'event_time'] = df.loc[mask_treated, 'event_time'].clip(
            -config.PRE_PERIODS, config.POST_PERIODS
        )
        
        df['event_time'] = pd.to_numeric(df['event_time'], errors='coerce').fillna(-999).astype(int)
        
        # Create leads and lags dummies
        for t in range(-config.PRE_PERIODS, config.POST_PERIODS + 1):
            if t < 0:
                dummy_name = f'lead_{abs(t)}'
            elif t == 0:
                dummy_name = 'treatment'
            else:
                dummy_name = f'lag_{t}'
            
            df[dummy_name] = 0
            mask = (df['first_treat_year'].notna()) & (df['event_time'] == t)
            df.loc[mask, dummy_name] = 1
            
            # Create interaction with treatment variable
            df[f'{dummy_name}_treat'] = df[dummy_name] * df[config.TREATMENT_VAR]
        
        # Create treatment indicator
        df['ever_treated'] = (~df['first_treat_year'].isna()).astype(int)
        
        logger.info(f"Created {config.PRE_PERIODS + config.POST_PERIODS + 1} event dummies")
        
        return df
        
    def save_individual_event_studies(self, results_df):
        """Save individual event study plots for key outcomes."""
        print("\n📊 Saving individual event study figures...")
        
        figure_map = {
            'log_real_wage_cpi': ('Figure14_event_study_log_wage.png', 'Log Real Wage'),
            'low_wage': ('Figure15_event_study_low_wage.png', 'Low-Wage Employment'),
            'informal': ('Figure16_event_study_informal.png', 'Informal Employment')
        }
        
        for outcome, (filename, title) in figure_map.items():
            if outcome not in results_df['outcome'].values:
                print(f"  ⚠️ {outcome} not found in results")
                continue
                
            outcome_result = next((r for r in self.results if r['outcome'] == outcome), None)
            if outcome_result is None:
                continue
                
            plot_df = outcome_result['results_df']
            
            plt.figure(figsize=(12, 7))
            
            pre = plot_df[plot_df['event_time'] < 0]
            post = plot_df[plot_df['event_time'] >= 0]
            
            # Scale: coefficient * 100 * 10 (for 0.1 bite increase)
            if len(pre) > 0:
                plt.errorbar(pre['event_time'], pre['coefficient'] * 100 * 10,
                            yerr=1.96 * pre['std_error'] * 100 * 10,
                            fmt='o-', color='blue', capsize=3, label='Pre-treatment', 
                            markersize=8, linewidth=2)
            
            if len(post) > 0:
                plt.errorbar(post['event_time'], post['coefficient'] * 100 * 10,
                            yerr=1.96 * post['std_error'] * 100 * 10,
                            fmt='s-', color='red', capsize=3, label='Post-treatment', 
                            markersize=8, linewidth=2)
            
            plt.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
            plt.axvline(x=-0.5, color='gray', linestyle='--', linewidth=2, alpha=0.7)
            
            #plt.title(f'Event Study: Effect on {title}', fontsize=14, fontweight='bold')
            plt.xlabel('Years Relative to Minimum Wage Increase', fontsize=12)
            plt.ylabel('Effect (percentage points)', fontsize=12)
            plt.legend(loc='best', frameon=True)
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(f"{config.FIGURES_DIR}/{filename}", dpi=300, bbox_inches='tight')
            plt.show()
            print(f"  ✓ Saved: {filename}") 

    def demean_data(self, df: pd.DataFrame, outcome: str, vars_to_demean: List[str]) -> pd.DataFrame:
        """
        Manually demean variables by individual (within transformation).
        Equivalent to including individual fixed effects.
        """
        df = df.copy()
        
        # Sort to ensure proper grouping
        df = df.sort_values(['idind', 'year'])
        
        # Demean each variable within individual
        for var in vars_to_demean:
            if var in df.columns:
                # Calculate individual means
                individual_mean = df.groupby('idind')[var].transform('mean')
                # Subtract mean
                df[f'{var}_demeaned'] = df[var] - individual_mean
            else:
                print(f"  Warning: {var} not found in data")
        
        return df
            
    def test_outcome(self, outcome: str, event_df: pd.DataFrame) -> Optional[Dict]:
        """Run event study with treatment intensity using demeaned data (individual FE via within transformation)."""
        
        if outcome not in event_df.columns:
            return None
        
        # Prepare data
        outcome_df = event_df.dropna(subset=[outcome, 'region', 'year', config.TREATMENT_VAR]).copy()
        
        if len(outcome_df) < 100:
            return None
        
        # Filter to relevant observations
        outcome_df = outcome_df[
            (outcome_df['ever_treated'] == 0) | 
            (outcome_df['event_time'].between(-config.PRE_PERIODS, config.POST_PERIODS))
        ].copy()
        
        # Build list of treat dummies
        treat_dummies = []
        for t in range(-config.PRE_PERIODS, config.POST_PERIODS + 1):
            if t < 0:
                name = f'lead_{abs(t)}_treat'
            elif t == 0:
                name = 'treatment_treat'
            else:
                name = f'lag_{t}_treat'
            if name in outcome_df.columns:
                treat_dummies.append(name)
        
        # Remove reference period (t=-1)
        reference = 'lead_1_treat'
        if reference in treat_dummies:
            treat_dummies.remove(reference)
        
        # Add control variables (will be demeaned)
        control_vars = []
        if 'age' in outcome_df.columns:
            outcome_df['age_sq'] = outcome_df['age'] ** 2
            control_vars.extend(['age', 'age_sq'])
        if 'female' in outcome_df.columns:
            control_vars.append('female')
        
        # List all variables to demean
        vars_to_demean = treat_dummies + control_vars
        if config.TREATMENT_VAR in outcome_df.columns and not config.TREATMENT_TIME_INVARIANT:
            vars_to_demean.append(config.TREATMENT_VAR)
        
        # Demean the outcome and all independent variables
        # First, ensure idind exists
        if 'idind' not in outcome_df.columns:
            logger.error("idind column not found for demeaning")
            return None
        
        # Demean outcome
        outcome_df = self.demean_data(outcome_df, outcome, vars_to_demean)
        
        # Also demean the outcome variable
        outcome_mean = outcome_df.groupby('idind')[outcome].transform('mean')
        outcome_df[f'{outcome}_demeaned'] = outcome_df[outcome] - outcome_mean
        
        # Build formula using demeaned variables
        demeaned_treat_dummies = [f'{d}_demeaned' for d in treat_dummies]
        formula = f"{outcome}_demeaned ~ " + " + ".join(demeaned_treat_dummies)
        
        # Add demeaned controls
        for var in control_vars:
            if f'{var}_demeaned' in outcome_df.columns:
                formula += f" + {var}_demeaned"
        
        # Add demeaned treatment main effect (if time-varying)
        if not config.TREATMENT_TIME_INVARIANT and f'{config.TREATMENT_VAR}_demeaned' in outcome_df.columns:
            formula += f" + {config.TREATMENT_VAR}_demeaned"
        
        # Add year fixed effects (still needed for common time shocks)
        formula += " + C(year)"
        
        try:
            # Run OLS on demeaned data (no individual FE needed - already removed)
            model = smf.ols(formula, data=outcome_df).fit(
                cov_type='cluster',
                cov_kwds={'groups': outcome_df['region']}
            )
            
            # Extract coefficients (same as before)
            results = []
            for t in range(-config.PRE_PERIODS, config.POST_PERIODS + 1):
                if t == -1:
                    continue
                    
                if t < 0:
                    name = f'lead_{abs(t)}_treat_demeaned'
                elif t == 0:
                    name = 'treatment_treat_demeaned'
                else:
                    name = f'lag_{t}_treat_demeaned'
                
                if name in model.params.index:
                    coef = model.params[name]
                    se = model.bse[name]
                    pval = model.pvalues[name]
                    ci_low, ci_high = model.conf_int().loc[name]
                    
                    results.append({
                        'event_time': t,
                        'coefficient': coef,
                        'std_error': se,
                        'p_value': pval,
                        'ci_lower': ci_low,
                        'ci_upper': ci_high
                    })
            
            if not results:
                return None
            
            results_df = pd.DataFrame(results).sort_values('event_time')
            
            # Pre-trends test (same as before)
            pre_data = results_df[results_df['event_time'] < 0]
            if len(pre_data) >= 2:
                pre_dummies = [name for name in model.params.index if 'lead_' in name and '_treat_demeaned' in name and 'lead_1' not in name]
                if len(pre_dummies) >= 2:
                    try:
                        f_test = model.f_test(pre_dummies)
                        pre_trend_p = f_test.pvalue
                    except Exception:
                        pre_trend_p = 1.0
                else:
                    pre_trend_p = 1.0
            else:
                pre_trend_p = 1.0
            
            # Post-treatment averages
            post_data = results_df[results_df['event_time'] >= 0]
            
            if len(post_data) > 0:
                avg_post = post_data['coefficient'].mean()
                avg_post_se = np.sqrt((post_data['std_error'] ** 2).mean())
                avg_post_p = 2 * (1 - stats.norm.cdf(abs(avg_post / avg_post_se)))
                
                lr_row = post_data[post_data['event_time'] == post_data['event_time'].max()]
                if len(lr_row) > 0:
                    lr_effect = lr_row['coefficient'].iloc[0]
                    lr_p = lr_row['p_value'].iloc[0]
                else:
                    lr_effect, lr_p = np.nan, np.nan
            else:
                avg_post, avg_post_p, lr_effect, lr_p = np.nan, np.nan, np.nan, np.nan
            
            treatment_scale = {
                'kaitz_median': 'per 0.1 increase in bite',
                'kaitz_mean': 'per 0.1 increase in bite',
                'kaitz_p25': 'per 0.1 increase in bite',
                'kaitz_p10': 'per 0.1 increase in bite',
                'mw_gap_1000': 'per 1000 ruble increase',
                'bite_share_2007': 'per 10pp increase in affected share'
            }.get(config.TREATMENT_VAR, 'per unit increase')
            
            return {
                'outcome': outcome,
                'treatment_var': config.TREATMENT_VAR,
                'treatment_scale': treatment_scale,
                'pre_trend_p': pre_trend_p,
                'pre_trend_slope': np.nan,
                'pre_trend_slope_p': np.nan,
                'avg_post': avg_post,
                'avg_post_p': avg_post_p,
                'lr_effect': lr_effect,
                'lr_p': lr_p,
                'n_obs': len(outcome_df),
                'model': model,
                'results_df': results_df
            }
            
        except Exception as e:
            logger.debug(f"Error testing {outcome}: {str(e)[:100]}")
            return None
   
    def test_all_outcomes(self) -> pd.DataFrame:
        """Test all outcomes and compile results."""
        
        self.define_outcomes()
        
        logger.info("Preparing event study data...")
        event_df = self.prepare_event_study_data()
        
        all_outcomes = []
        for category, outcomes in self.outcome_categories.items():
            all_outcomes.extend(outcomes)
        
        logger.info(f"Testing {len(all_outcomes)} outcomes...")
        
        for i, outcome in enumerate(all_outcomes):
            if i % 10 == 0:
                logger.info(f"  Progress: {i}/{len(all_outcomes)}")
            
            result = self.test_outcome(outcome, event_df)
            if result:
                for cat, outs in self.outcome_categories.items():
                    if outcome in outs:
                        result['category'] = cat
                        break
                
                self.results.append(result)
        
        results_df = pd.DataFrame(self.results)
        
        if len(results_df) > 0:
            p_values = results_df['avg_post_p'].fillna(1).values
            _, p_corrected, _, _ = multipletests(
                p_values, alpha=config.ALPHA, method='fdr_bh'
            )
            results_df['avg_post_p_corrected'] = p_corrected
            results_df['significant'] = p_corrected < config.ALPHA
            
            if any(x in config.TREATMENT_VAR for x in ['kaitz', 'bite']):
                scale_factor = 0.1
            else:
                scale_factor = 1
            
            results_df['avg_post_scaled'] = results_df['avg_post'] * scale_factor
            results_df['lr_effect_scaled'] = results_df['lr_effect'] * scale_factor
            results_df['effect_magnitude_pp'] = results_df['avg_post_scaled'] * 100
            results_df['lr_magnitude_pp'] = results_df['lr_effect_scaled'] * 100
            
            results_df['effect_direction'] = results_df['avg_post'].apply(
                lambda x: 'Positive' if x > 0 else 'Negative' if x < 0 else 'Zero'
            )
        
        return results_df
    
    def create_summary_tables(self, results_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Create summary tables for output."""
        
        tables = {}
        
        table1 = results_df[[
            'category', 'outcome', 'avg_post_scaled', 'avg_post_p', 
            'avg_post_p_corrected', 'significant', 'pre_trend_p',
            'lr_effect_scaled', 'lr_p', 'n_obs', 'treatment_scale'
        ]].copy()
        
        table1['avg_post_pp'] = table1['avg_post_scaled'] * 100
        table1['lr_effect_pp'] = table1['lr_effect_scaled'] * 100
        table1['avg_post_formatted'] = table1.apply(
            lambda row: f"{row['avg_post_pp']:.2f}pp {format_pvalue(row['avg_post_p'], include_stars=False)}",
            axis=1
        )
        table1['pre_trend_formatted'] = table1['pre_trend_p'].apply(
            lambda p: f"{p:.4f}" if not pd.isna(p) else "N/A"
        )
        
        tables['all_outcomes'] = table1
        
        sig_results = results_df[results_df['significant']].copy()
        if len(sig_results) > 0:
            sig_results = sig_results.sort_values('avg_post_p_corrected')
            tables['significant_outcomes'] = sig_results
        
        strong_lr = results_df[
            (results_df['lr_p'].fillna(1) < 0.10) & 
            (abs(results_df['lr_effect_scaled']).fillna(0) > 0.01)
        ].copy()
        if len(strong_lr) > 0:
            strong_lr = strong_lr.sort_values('lr_p')
            tables['strong_longrun'] = strong_lr
        
        cat_summary = results_df.groupby('category').agg({
            'outcome': 'count',
            'significant': 'sum',
            'avg_post_scaled': lambda x: x.mean() * 100,
            'avg_post_p': lambda x: (x < 0.05).sum() / len(x) * 100
        }).round(2)
        cat_summary.columns = ['N Outcomes', 'N Significant', 'Avg Effect (pp)', '% Significant (p<0.05)']
        tables['category_summary'] = cat_summary
        
        return tables
    
    def plot_top_outcomes(self, results_df: pd.DataFrame, n_top: int = 6) -> None:
        """Plot event studies for top outcomes."""
        
        top_outcomes = results_df.nsmallest(n_top, 'avg_post_p_corrected')
        
        n_cols = 3
        n_rows = (n_top + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))
        axes = axes.flatten()
        
        for i, (_, row) in enumerate(top_outcomes.iterrows()):
            if i >= len(axes):
                break
            
            outcome_result = next(r for r in self.results if r['outcome'] == row['outcome'])
            plot_df = outcome_result['results_df']
            
            ax = axes[i]
            
            pre = plot_df[plot_df['event_time'] < 0]
            post = plot_df[plot_df['event_time'] >= 0]
            
            if any(x in config.TREATMENT_VAR for x in ['kaitz', 'bite']):
                scale = 0.1 * 100
            else:
                scale = 100
            
            if len(pre) > 0:
                ax.errorbar(pre['event_time'], pre['coefficient'] * scale,
                           yerr=1.96 * pre['std_error'] * scale,
                           fmt='o-', color=config.COLOR_PRE, capsize=3,
                           label='Pre-treatment', markersize=6)
            
            if len(post) > 0:
                ax.errorbar(post['event_time'], post['coefficient'] * scale,
                           yerr=1.96 * post['std_error'] * scale,
                           fmt='s-', color=config.COLOR_POST, capsize=3,
                           label='Post-treatment', markersize=6)
            
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
            ax.axvline(x=-0.5, color=config.COLOR_REF, linestyle='--', linewidth=1, alpha=0.7)
            
            pre_trend_p = row['pre_trend_p']
            if not pd.isna(pre_trend_p):
                color = 'green' if pre_trend_p > 0.05 else 'red'
                ax.text(0.02, 0.98, f'Pre-trend p={pre_trend_p:.4f}',
                       transform=ax.transAxes, color=color, fontsize=9,
                       verticalalignment='top')
            
            treatment_label = {
                'kaitz_median': ' (per 0.1↑ bite)',
                'kaitz_mean': ' (per 0.1↑ bite)',
                'kaitz_p25': ' (per 0.1↑ bite)',
                'kaitz_p10': ' (per 0.1↑ bite)',
                'mw_gap_1000': ' (per 1000₽)',
                'bite_share_2007': ' (per 10pp↑)'
            }.get(config.TREATMENT_VAR, '')
            
            #ax.set_title(f"{row['outcome'].replace('_', ' ').title()}{treatment_label}", fontsize=11)
            ax.set_xlabel('Years Relative to Treatment', fontsize=9)
            ax.set_ylabel('Effect (pp)', fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8, loc='lower right')
        
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        
        #plt.suptitle(f'Top Outcomes: Event Study Results ({config.TREATMENT_VAR})',fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        save_figure(plt, f'top_outcomes_{config.TREATMENT_VAR}.png')
        plt.show()


# ============================================================================
# ANALYSIS PIPELINE
# ============================================================================

class AnalysisPipeline:
    """Main pipeline with proven methodology."""
    
    def __init__(self, df_final: pd.DataFrame):
        self.df_final = df_final.copy()
        self.outcome_tester = OutcomeTester(self.df_final)
        self.results = {}
        self.tables = {}
        
    def run_all(self) -> None:
        """Run complete analysis pipeline."""
        logger.info("="*80)
        logger.info("STARTING METHODOLOGY ANALYSIS")
        logger.info(f"Treatment variable: {config.TREATMENT_VAR}")
        logger.info("="*80)
        
        # Step 1: Test all outcomes
        logger.info("\n📊 Step 1: Testing all outcomes...")
        results_df = self.outcome_tester.test_all_outcomes()
        
        if len(results_df) == 0:
            logger.error("No results generated!")
            return
        
        self.results['full'] = results_df
        
        # Step 2: Create summary tables
        logger.info("\n📊 Step 2: Creating summary tables...")
        self.tables = self.outcome_tester.create_summary_tables(results_df)
        
        # Step 3: Save all tables
        logger.info("\n📊 Step 3: Saving tables...")
        for name, table in self.tables.items():
            save_table(table, f"table_{name}_{config.TREATMENT_VAR}")
        
        # Step 4: Generate individual-level plots
        logger.info("\n📊 Step 4: Generating individual-level plots...")
        self.outcome_tester.plot_top_outcomes(results_df) 
        self.outcome_tester.save_individual_event_studies(results_df)
        
        # Step 5: Print summary
        self._print_summary()
        
        logger.info("\n" + "="*80)
        logger.info("✅ ANALYSIS COMPLETE")
        logger.info("="*80)
    
    def _print_summary(self) -> None:
        """Print analysis summary."""
        
        results_df = self.results['full']
        
        print("\n" + "="*100)
        print(f"ANALYSIS SUMMARY - Treatment: {config.TREATMENT_VAR}")
        print("="*100)
        
        print(f"\n📊 Total outcomes tested: {len(results_df)}")
        print(f"   Significant after FDR correction: {results_df['significant'].sum()}")
        
        print("\n📈 Effect size distribution (scaled for interpretation):")
        if 'effect_magnitude_pp' in results_df.columns:
            effect_sizes = results_df['effect_magnitude_pp'].dropna()
            print(f"   Mean: {effect_sizes.mean():.2f}pp")
            print(f"   Std:  {effect_sizes.std():.2f}pp")
            print(f"   Min:  {effect_sizes.min():.2f}pp")
            print(f"   Max:  {effect_sizes.max():.2f}pp")
        
        print("\n📋 Results by category:")
        if 'category_summary' in self.tables:
            cat_summary = self.tables['category_summary']
            print(tabulate(cat_summary, headers='keys', tablefmt='simple'))
        
        if 'significant_outcomes' in self.tables:
            sig_table = self.tables['significant_outcomes']
            print("\n🎯 Significant outcomes (FDR q<0.05):")
            for _, row in sig_table.iterrows():
                effect_pp = row.get('effect_magnitude_pp', row['avg_post_scaled'] * 100)
                scale_text = row.get('treatment_scale', 'per unit')
                print(f"   • {row['outcome']}: {effect_pp:.2f}pp {scale_text} (p={row['avg_post_p_corrected']:.4f})")
        
        if 'strong_longrun' in self.tables:
            lr_table = self.tables['strong_longrun']
            print("\n💪 Strong long-run effects (p<0.10, |effect|>1pp):")
            for _, row in lr_table.iterrows():
                pre_warn = "⚠️" if row['pre_trend_p'] < 0.05 else "✅"
                lr_pp = row.get('lr_magnitude_pp', row['lr_effect_scaled'] * 100)
                scale_text = row.get('treatment_scale', 'per unit')
                print(f"   {pre_warn} {row['outcome']}: {lr_pp:.2f}pp {scale_text} (p={row['lr_p']:.4f})")
        
        print("\n" + "="*100)
        print(f"📁 Tables saved to: {config.TABLES_DIR}/")
        print(f"📁 Figures saved to: {config.FIGURES_DIR}/")
        print("="*100)
 
# ============================================================================
# ADDITIONAL FIGURES FOR THESIS NARRATIVE
# ============================================================================

class ThesisFigures:
    """Generate additional figures to support thesis narrative."""
    
    def __init__(self, df: pd.DataFrame, outcome_tester: OutcomeTester, results_df: pd.DataFrame):
        self.df = df
        self.outcome_tester = outcome_tester
        self.results_df = results_df
        self.fig_dir = config.FIGURES_DIR
        
    def generate_all(self):
        """Generate all thesis figures."""
        print("\n" + "="*80)
        print("GENERATING THESIS NARRATIVE FIGURES")
        print("="*80)
        
        self.figure_causal_summary()
        self.figure_within_vs_between()
        self.figure_pre_trends_dashboard()
        self.figure_results_heatmap()
        self.figure_adjustment_mechanisms()
        self.figure_causal_confidence()
        
        print("\n✅ All thesis narrative figures generated!")
        print(f"   Saved to: {self.fig_dir}/")
    
    def figure_causal_summary(self):
        """Figure 17: Causal Effects Summary - Your cleanest results"""
        
        # Extract key causal outcomes (those with pre-trend p > 0.05 and significant)
        causal_outcomes = self.results_df[
            (self.results_df['pre_trend_p'] > 0.05) & 
            (self.results_df['significant'] == True)
        ].copy()
        
        if len(causal_outcomes) == 0:
            print("  ⚠️ No causal outcomes found with pre-trend p>0.05")
            return
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        outcomes = causal_outcomes['outcome'].tolist()
        effects = causal_outcomes['effect_magnitude_pp'].tolist()
        errors = causal_outcomes.apply(
            lambda row: abs(row['avg_post_scaled'] * 100 * 0.3),  # Approx SE
            axis=1
        ).tolist()
        
        # Color code: green for positive, red for negative
        colors = ['#44aa44' if e > 0 else '#ff4444' for e in effects]
        
        y_pos = np.arange(len(outcomes))
        bars = ax.barh(y_pos, effects, xerr=errors, color=colors, 
                      alpha=0.8, edgecolor='black', linewidth=1.5,
                      capsize=5, error_kw={'linewidth': 2})
        
        # Add value labels
        for i, (bar, effect, error) in enumerate(zip(bars, effects, errors)):
            label_x = effect + error + 0.5 if effect > 0 else effect - error - 1.5
            ax.text(label_x, bar.get_y() + bar.get_height()/2,
                   f'{effect:.1f}pp', ha='center', va='center',
                   fontweight='bold', fontsize=11,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9))
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels([o.replace('_', ' ').title() for o in outcomes], fontsize=12)
        ax.set_xlabel('Effect Size (percentage points)', fontweight='bold', fontsize=14)
        #ax.set_title('Figure 17: Causal Effects of Minimum Wage Increase\n(Parallel Trends Hold, FDR q<0.05)', fontweight='bold', fontsize=16)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add annotation about interpretation
        treatment_scale = self.results_df['treatment_scale'].iloc[0]
        ax.text(0.98, 0.02, f'Effect shown: {treatment_scale}', 
               transform=ax.transAxes, ha='right', fontsize=10,
               bbox=dict(boxstyle="round", facecolor='lightyellow', alpha=0.8))
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure17_causal_summary.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure17_causal_summary.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure17_causal_summary.png")
    
    def figure_within_vs_between(self):
        """Figure 18: Within-Occupation vs Between-Occupation Adjustment"""
        
        # Group outcomes by adjustment channel
        within_occupation = [
            'log_real_wage_cpi', 'informal', 'low_wage', 'below_min_wage'
        ]
        between_occupation = [
            'occ_changed_4digit', 'occ_changed_1digit', 
            'upward_mobility', 'downward_mobility', 'lateral_mobility'
        ]
        
        # Calculate average effects
        within_effects = []
        for outcome in within_occupation:
            row = self.results_df[self.results_df['outcome'] == outcome]
            if len(row) > 0:
                within_effects.append(abs(row['effect_magnitude_pp'].iloc[0]))
        
        between_effects = []
        for outcome in between_occupation:
            row = self.results_df[self.results_df['outcome'] == outcome]
            if len(row) > 0:
                between_effects.append(abs(row['effect_magnitude_pp'].iloc[0]))
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Left: Average effect magnitude
        avg_within = np.mean(within_effects) if within_effects else 0
        avg_between = np.mean(between_effects) if between_effects else 0
        
        bars = ax1.bar(['Within Occupation', 'Between Occupation'], 
                      [avg_within, avg_between],
                      color=['#2ca02c', '#d62728'], alpha=0.8,
                      edgecolor='black', linewidth=1.5)
        
        for bar, val in zip(bars, [avg_within, avg_between]):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.2,
                    f'{val:.2f}pp', ha='center', va='bottom',
                    fontweight='bold', fontsize=12)
        
        ax1.set_ylabel('Average Effect Magnitude (pp)', fontweight='bold', fontsize=12)
        ax1.set_title('(a) Where Does Adjustment Happen?', fontweight='bold', fontsize=14)
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Right: Share of significant outcomes
        within_sig = sum(1 for o in within_occupation 
                        if len(self.results_df[self.results_df['outcome'] == o]) > 0 and
                        self.results_df[self.results_df['outcome'] == o]['significant'].iloc[0])
        within_total = len([o for o in within_occupation 
                          if o in self.results_df['outcome'].values])
        
        between_sig = sum(1 for o in between_occupation 
                         if len(self.results_df[self.results_df['outcome'] == o]) > 0 and
                         self.results_df[self.results_df['outcome'] == o]['significant'].iloc[0])
        between_total = len([o for o in between_occupation 
                           if o in self.results_df['outcome'].values])
        
        sig_share_within = (within_sig / within_total * 100) if within_total > 0 else 0
        sig_share_between = (between_sig / between_total * 100) if between_total > 0 else 0
        
        bars2 = ax2.bar(['Within Occupation', 'Between Occupation'],
                        [sig_share_within, sig_share_between],
                        color=['#2ca02c', '#d62728'], alpha=0.8,
                        edgecolor='black', linewidth=1.5)
        
        for bar, val in zip(bars2, [sig_share_within, sig_share_between]):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
                    f'{val:.0f}%', ha='center', va='bottom',
                    fontweight='bold', fontsize=12)
        
        ax2.set_ylabel('Significant Outcomes (%)', fontweight='bold', fontsize=12)
        ax2.set_title('(b) Share of Outcomes Significant', fontweight='bold', fontsize=14)
        ax2.grid(True, alpha=0.3, axis='y')
        
        #fig.suptitle('Figure 18: Adjustment Occurs Within Occupations', fontweight='bold', fontsize=16, y=1.02)
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure18_within_vs_between.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure18_within_vs_between.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure18_within_vs_between.png")
    
    def figure_pre_trends_dashboard(self):
        """Figure 19: Pre-Trends Dashboard - Transparency about identification"""
        
        # Select key outcomes
        key_outcomes = [
            'log_real_wage_cpi', 'low_wage', 'informal', 
            'below_min_wage', 'exit_professional', 'occ_changed_4digit'
        ]
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, outcome in enumerate(key_outcomes):
            if idx >= len(axes):
                break
                
            ax = axes[idx]
            
            # Find outcome result
            outcome_result = next((r for r in self.outcome_tester.results 
                                 if r['outcome'] == outcome), None)
            
            if outcome_result is None:
                ax.text(0.5, 0.5, f'{outcome}\nNot found', ha='center', va='center', fontsize=12)
                #ax.set_title(outcome.replace('_', ' ').title(), fontweight='bold')
                continue
            
            plot_df = outcome_result['results_df']
            pre = plot_df[plot_df['event_time'] < 0]
            post = plot_df[plot_df['event_time'] >= 0]
            
            # Plot pre-trend coefficients
            if len(pre) > 0:
                ax.errorbar(pre['event_time'], pre['coefficient'] * 100 * 10,
                           yerr=1.96 * pre['std_error'] * 100 * 10,
                           fmt='o-', color='blue', capsize=3,
                           label='Pre-treatment', markersize=6)
            
            # Plot post-treatment
            if len(post) > 0:
                ax.errorbar(post['event_time'], post['coefficient'] * 100 * 10,
                           yerr=1.96 * post['std_error'] * 100 * 10,
                           fmt='s-', color='red', capsize=3,
                           label='Post-treatment', markersize=6)
            
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
            ax.axvline(x=-0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
            
            # Get pre-trend p-value
            row = self.results_df[self.results_df['outcome'] == outcome]
            pre_trend_p = row['pre_trend_p'].iloc[0] if len(row) > 0 else 1.0
            
            # Color code based on pre-trend test
            if pre_trend_p > 0.05:
                color = 'green'
                status = '✅ Parallel trends hold'
            else:
                color = 'red'
                status = '⚠️ Pre-trends present'
            
            ax.text(0.02, 0.98, f'p={pre_trend_p:.4f}\n{status}', 
                   transform=ax.transAxes, color=color, fontsize=9,
                   verticalalignment='top',
                   bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
            
            #ax.set_title(outcome.replace('_', ' ').title(), fontweight='bold', fontsize=11)
            ax.set_xlabel('Years', fontsize=8)
            ax.set_ylabel('Effect (pp)', fontsize=8)
            ax.grid(True, alpha=0.3)
            
            if idx == 0:
                ax.legend(fontsize=8, loc='lower right')
        
        #plt.suptitle('Figure 19: Pre-Trends Diagnostic - Key Outcomes', fontweight='bold', fontsize=16, y=1.02)
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure19_pre_trends_dashboard.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure19_pre_trends_dashboard.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure19_pre_trends_dashboard.png")
    
    def figure_results_heatmap(self):
        """Figure 20: Results Summary Heatmap"""
        
        # Select outcomes for heatmap
        outcomes_to_show = [
            'log_real_wage_cpi', 'low_wage', 'informal', 
            'below_min_wage', 'exit_professional', 'occ_changed_4digit',
            'upward_mobility', 'downward_mobility', 'professional'
        ]
        
        # Filter to outcomes that exist
        plot_outcomes = []
        effect_sizes = []
        p_values = []
        pre_trend_p = []
        
        for outcome in outcomes_to_show:
            row = self.results_df[self.results_df['outcome'] == outcome]
            if len(row) > 0:
                plot_outcomes.append(outcome)
                effect_sizes.append(row['effect_magnitude_pp'].iloc[0])
                p_values.append(row['avg_post_p_corrected'].iloc[0])
                pre_trend_p.append(row['pre_trend_p'].iloc[0])
        
        if not plot_outcomes:
            return
        
        # Create matrices for heatmap
        data = np.array([
            effect_sizes,
            [-np.log10(p) if p > 0 else 3 for p in p_values],  # -log10 p-value
            [1 if p > 0.05 else 0 for p in pre_trend_p]  # 1 if parallel trends hold
        ])
        
        fig, ax = plt.subplots(figsize=(12, 5))
        
        # Create custom colormap
        im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=-5, vmax=5)
        
        # Set labels
        ax.set_xticks(np.arange(len(plot_outcomes)))
        ax.set_yticks([0, 1, 2])
        ax.set_xticklabels([o.replace('_', ' ').title() for o in plot_outcomes], 
                          rotation=45, ha='right', fontsize=10)
        ax.set_yticklabels(['Effect Size (pp)', 'Significance\n(-log10 p)', 'Parallel Trends\n(1=Yes)'], 
                          fontweight='bold', fontsize=10)
        
        # Add text annotations
        for i in range(3):
            for j in range(len(plot_outcomes)):
                if i == 0:
                    text = f'{data[i,j]:.1f}'
                elif i == 1:
                    text = f'{data[i,j]:.1f}'
                else:
                    text = '✓' if data[i,j] == 1 else '✗'
                
                ax.text(j, i, text, ha='center', va='center',
                       fontweight='bold', fontsize=10,
                       color='white' if abs(data[i,j]) > 2 else 'black')
        
        #plt.title('Figure 20: Results Summary - Effect Sizes, Significance, and Identification',fontweight='bold', fontsize=16, pad=20)
        plt.colorbar(im, ax=ax, label='Effect Size (pp) / -log10(p)')
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure20_results_heatmap.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure20_results_heatmap.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure20_results_heatmap.png")
    
    def figure_adjustment_mechanisms(self):
        """Figure 21: Adjustment Mechanisms - How workers respond"""
        
        # Group outcomes by mechanism
        mechanisms = {
            'Wage Gains': ['log_real_wage_cpi'],
            'Formalization': ['informal'],
            'Low-Wage Exit': ['low_wage'],
            'Professional Exit': ['exit_professional'],
            'Occupational Mobility': ['occ_changed_4digit', 'occ_changed_1digit'],
            'Gender Segregation': ['stayed_male', 'stayed_female'] if 'stayed_male' in self.results_df['outcome'].values else []
        }
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        mechanism_names = []
        avg_effects = []
        colors = []
        
        for mech_name, outcomes in mechanisms.items():
            effects = []
            for outcome in outcomes:
                row = self.results_df[self.results_df['outcome'] == outcome]
                if len(row) > 0:
                    effects.append(abs(row['effect_magnitude_pp'].iloc[0]))
            
            if effects:
                mechanism_names.append(mech_name)
                avg_effects.append(np.mean(effects))
                
                # Color based on mechanism type
                if mech_name in ['Wage Gains', 'Formalization', 'Low-Wage Exit']:
                    colors.append('#2ca02c')  # Green - within-occupation
                elif mech_name == 'Professional Exit':
                    colors.append('#ff7f0e')  # Orange - mixed
                else:
                    colors.append('#d62728')  # Red - no effect/between
        
        y_pos = np.arange(len(mechanism_names))
        bars = ax.barh(y_pos, avg_effects, color=colors, alpha=0.8,
                      edgecolor='black', linewidth=1.5)
        
        # Add value labels
        for bar, effect in zip(bars, avg_effects):
            ax.text(effect + 0.1, bar.get_y() + bar.get_height()/2,
                   f'{effect:.2f}pp', ha='left', va='center',
                   fontweight='bold', fontsize=11)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(mechanism_names, fontweight='bold', fontsize=12)
        ax.set_xlabel('Average Effect Magnitude (percentage points)', fontweight='bold', fontsize=14)
        #ax.set_title('Figure 21: Adjustment Mechanisms - How Workers Respond',  fontweight='bold', fontsize=16)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add annotation
        ax.text(0.98, 0.02, 'Green = Within-Occupation Adjustment\nRed = No Effect/Between', 
               transform=ax.transAxes, ha='right', fontsize=10,
               bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure21_adjustment_mechanisms.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure21_adjustment_mechanisms.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure21_adjustment_mechanisms.png")
    
    def figure_causal_confidence(self):
        """Figure 22: Confidence in Causal Claims"""
        
        # Classify outcomes by causal confidence
        causal_confident = []  # pre-trend p>0.05 & significant
        causal_suggestive = []  # pre-trend p<0.05 but significant & large
        null_clear = []  # not significant, well-powered
        
        for _, row in self.results_df.iterrows():
            outcome = row['outcome']
            effect = abs(row['effect_magnitude_pp'])
            sig = row['significant']
            pre_p = row['pre_trend_p']
            
            if sig and pre_p > 0.05:
                causal_confident.append(outcome)
            elif sig and pre_p <= 0.05:
                causal_suggestive.append(outcome)
            elif not sig and effect < 1.0:
                null_clear.append(outcome)
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(10, 6))
        
        categories = ['Causal\n(Parallel Trends + Significant)', 
                      'Suggestive\n(Significant but Pre-Trends)',
                      'Null\n(No Effect, Well-Powered)']
        counts = [len(causal_confident), len(causal_suggestive), len(null_clear)]
        colors = ['#2ca02c', '#ff7f0e', '#1f77b4']
        
        bars = ax.bar(categories, counts, color=colors, alpha=0.8,
                     edgecolor='black', linewidth=2)
        
        # Add value labels
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                   f'{count} outcomes', ha='center', va='bottom',
                   fontweight='bold', fontsize=12)
        
        # Add example outcomes
        if causal_confident:
            examples = ', '.join([o.replace('_', ' ').title()[:15] for o in causal_confident[:2]])
            ax.text(0, counts[0]/2, f'e.g., {examples}', 
                   ha='center', va='center', fontsize=9,
                   bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        ax.set_ylabel('Number of Outcomes', fontweight='bold', fontsize=14)
        #ax.set_title('Figure 22: Confidence in Causal Claims', fontweight='bold', fontsize=16)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(self.fig_dir / "Figure22_causal_confidence.png", dpi=300, bbox_inches='tight')
        plt.savefig(self.fig_dir / "Figure22_causal_confidence.pdf", bbox_inches='tight')
        plt.show()
        print("  ✓ Saved: Figure22_causal_confidence.png")


# ============================================================================
# MODIFIED MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function with thesis figures."""
    
    # Run pipeline
    pipeline = AnalysisPipeline(df_final)
    pipeline.run_all()
    
    # Generate thesis narrative figures
    if pipeline.results and 'full' in pipeline.results:
        thesis_figs = ThesisFigures(
            df_final, 
            pipeline.outcome_tester, 
            pipeline.results['full']
        )
        thesis_figs.generate_all()


if __name__ == "__main__":
    main() 